# Chapter 29 — Applied AI

**Companion to Applied AI**

Question: Does the whole machine hold together in miniature — and refuse four corruptions?

By the end of this notebook you will have:

- assembled task-to-acceptance end to end with mocks only
- passed six durable stops: decision, call, action, check, accept, replay
- broke grant, target, key, and crash — and shown refused state each time

## What this notebook demonstrates
The completed machine in miniature: context compilation → deterministic route → stochastic proposal → raw preservation → normalization → verification → authority check → effect → recorded evidence. Mocks only; no API key.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
import hashlib, json, time, uuid

seed: 42


## 1. The spine: six durable stops

In [2]:
ledger = []
def stop(kind: str, data: dict):
    ledger.append({"kind": kind, **data})

task = {"task_id": "t-final", "goal": "publish deploy note"}
stop("context", {"selected": ["spec", "last-run"], "package_id": "pkg-77e2"})
stop("route", {"chamber": "draft", "rule": "R1", "used_model_call": False})
stop("call", {"call_id": "c-1", "attempts": 1})
stop("raw", {"call_id": "c-1", "bytes": b'{"note": "canary 10%"}'})
stop("check", {"call_id": "c-1", "verdict": "pass"})
stop("accept", {"task_id": "t-final", "by": "verifier"})
print("stops:", [e["kind"] for e in ledger])
assert [e["kind"] for e in ledger] == ["context", "route", "call", "raw", "check", "accept"]

stops: ['context', 'route', 'call', 'raw', 'check', 'accept']


## 2. Effect only through grant, once per key

In [3]:
effects, keys = [], set()
def do_effect(key: str, grant: bool):
    if not grant:
        return "REFUSED: no grant"
    if key in keys:
        return "REFUSED: replay, 1 effect total"
    keys.add(key)
    effects.append(key)
    return "EFFECTED"

print(do_effect("k-final", True))
print(do_effect("k-final", True))
assert len(effects) == 1 and sum(1 for e in effects) == 1

EFFECTED
REFUSED: replay, 1 effect total


## 3. Four corruptions, four refusals

In [4]:
def guarded(step: str, **kw):
    if step == "grant" and not kw.get("grant"):
        return "refused: no grant"
    if step == "target" and kw.get("target") != "sandbox/note.txt":
        return "refused: target outside grant"
    if step == "key" and kw.get("key") in keys:
        return "refused: duplicate key"
    if step == "crash" and kw.get("phase") == "in-flight":
        return "refused: resume requires reconciliation"
    return "ok"

for name, kw in [("grant", {"grant": False}), ("target", {"target": "/prod/db"}),
                 ("key", {"key": "k-final"}), ("crash", {"phase": "in-flight"})]:
    r = guarded(name, **kw)
    print(f"corrupt {name:7s} -> {r}")
    assert r.startswith("refused")

corrupt grant   -> refused: no grant
corrupt target  -> refused: target outside grant
corrupt key     -> refused: duplicate key
corrupt crash   -> refused: resume requires reconciliation


## Interpretation
- Supports: composition holds for one path with durable stops; decision→action, grant→execution, effect→acceptance stay conventional even when proposals are stochastic.
- Does NOT support: production readiness; this is clarity, not scale.

## Connection to CodeAI
Conceptually: `context` ↔ context compilation, `route` ↔ the model-free router, `call/raw` ↔ recorded model calls, `check/accept` ↔ verification and acceptance. The notebook is useful without CodeAI installed.

## Try it yourself
1. Add a seventh stop: `replay` after a crash and reconcile from the ledger.
2. Corrupt the raw bytes and require v2 reinterpretation before acceptance.
3. Price the whole path with the chapter-06 calculator.